# Lista 4 — Zeros de Funções

Notebook preparado para rodar no **Jupyter**, com:
- enunciados reescritos;
- contexto das funções;
- resolução numérica;
- comparações entre métodos.

Sempre que possível, a solução numérica é acompanhada de uma interpretação do comportamento do método.

In [1]:
import numpy as np
import sympy as sp

np.set_printoptions(precision=10, suppress=True)

## Contexto das funções utilizadas

Funções genéricas implementadas:

- `newton(f, df, x0)`: Newton-Raphson;
- `newton_modified_m(f, df, x0, m)`: Newton modificado quando a multiplicidade é conhecida;
- `secant(f, x0, x1)`: método da secante;
- `modified_secant(f, x0, delta)`: secante modificada com perturbação relativa;
- `newton_system(F, J, x0)`: Newton para sistemas não lineares;
- `fixed_point_system(G, x0)`: iteração de ponto fixo para sistemas.

Essas rotinas foram escritas de forma reutilizável.

In [2]:
def newton(f, df, x0, tol=1e-6, maxit=100):
    x = float(x0)
    hist = [x]
    for k in range(1, maxit + 1):
        x_new = x - f(x)/df(x)
        hist.append(x_new)
        if abs(x_new - x) < tol or abs(f(x_new)) < tol:
            return x_new, k, np.array(hist, dtype=float)
        x = x_new
    return x, maxit, np.array(hist, dtype=float)

def newton_modified_m(f, df, x0, m, tol=1e-6, maxit=100):
    x = float(x0)
    hist = [x]
    for k in range(1, maxit + 1):
        x_new = x - m*f(x)/df(x)
        hist.append(x_new)
        if abs(x_new - x) < tol or abs(f(x_new)) < tol:
            return x_new, k, np.array(hist, dtype=float)
        x = x_new
    return x, maxit, np.array(hist, dtype=float)

def secant(f, x0, x1, tol=1e-6, maxit=100):
    x0 = float(x0)
    x1 = float(x1)
    hist = [x0, x1]
    for k in range(2, maxit + 2):
        fx0, fx1 = f(x0), f(x1)
        x2 = x1 - fx1*(x1 - x0)/(fx1 - fx0)
        hist.append(x2)
        if abs(x2 - x1) < tol or abs(f(x2)) < tol:
            return x2, k - 1, np.array(hist, dtype=float)
        x0, x1 = x1, x2
    return x1, maxit, np.array(hist, dtype=float)

def modified_secant(f, x0, delta=0.01, tol=1e-6, maxit=100):
    x = float(x0)
    hist = [x]
    for k in range(1, maxit + 1):
        denom = f(x + delta*x) - f(x)
        x_new = x - delta*x*f(x)/denom
        hist.append(x_new)
        if abs(x_new - x) < tol or abs(f(x_new)) < tol:
            return x_new, k, np.array(hist, dtype=float)
        x = x_new
    return x, maxit, np.array(hist, dtype=float)

def newton_system(F, J, x0, tol=1e-6, maxit=50):
    x = np.asarray(x0, dtype=float)
    hist = [x.copy()]
    for k in range(1, maxit + 1):
        delta = np.linalg.solve(J(x), -F(x))
        x_new = x + delta
        hist.append(x_new.copy())
        if np.linalg.norm(delta, np.inf) < tol or np.linalg.norm(F(x_new), np.inf) < tol:
            return x_new, k, np.array(hist, dtype=float)
        x = x_new
    return x, maxit, np.array(hist, dtype=float)

def fixed_point_system(G, x0, tol=1e-6, maxit=10000):
    x = np.asarray(x0, dtype=float)
    hist = [x.copy()]
    for k in range(1, maxit + 1):
        x_new = np.asarray(G(x), dtype=float)
        hist.append(x_new.copy())
        if np.linalg.norm(x_new - x, np.inf) < tol:
            return x_new, k, np.array(hist, dtype=float)
        x = x_new
    return x, maxit, np.array(hist, dtype=float)

## Questão 1

**Enunciado reescrito.** Para
\[
f(s) = s^3 + 7s^2 + 16s + 12,
\]
analisar:
- a raiz simples com a **secante modificada**, usando dois valores de \(\delta\);
- a raiz múltipla com Newton padrão e Newton modificado.

Fatorando:
\[
f(s) = (s+2)^2(s+3).
\]
Logo:
- raiz simples: \(s=-3\);
- raiz dupla: \(s=-2\).

In [4]:
f = lambda s: s**3 + 7*s**2 + 16*s + 12
df = lambda s: 3*s**2 + 14*s + 16

print("Raiz simples s = -3")
for delta in [0.01, 0.05]:
    raiz, it, hist = modified_secant(f, -3.5, delta=delta, tol=1e-6)
    print(f"delta = {delta:.2f} -> raiz ≈ {raiz:.10f} | iterações = {it}")

print("\nRaiz múltipla s = -2")
raiz_newton, it_newton, _ = newton(f, df, -1.5, tol=1e-6)
raiz_mod, it_mod, _ = newton_modified_m(f, df, -1.5, m=2, tol=1e-6)

print(f"Newton padrão      -> raiz ≈ {raiz_newton:.10f} | iterações = {it_newton}")
print(f"Newton modificado  -> raiz ≈ {raiz_mod:.10f} | iterações = {it_mod}")

print("\nInterpretação:")
print("- Na raiz múltipla, o Newton padrão perde velocidade de convergência.")
print("- O Newton modificado recupera a convergência mais rápida quando a multiplicidade é conhecida.")

Raiz simples s = -3
delta = 0.01 -> raiz ≈ -3.0000001260 | iterações = 7
delta = 0.05 -> raiz ≈ -3.0000004983 | iterações = 11

Raiz múltipla s = -2
Newton padrão      -> raiz ≈ -1.9992944468 | iterações = 10
Newton modificado  -> raiz ≈ -1.9999973546 | iterações = 3

Interpretação:
- Na raiz múltipla, o Newton padrão perde velocidade de convergência.
- O Newton modificado recupera a convergência mais rápida quando a multiplicidade é conhecida.


## Questão 2

**Enunciado reescrito.** Resolver o sistema não linear:
\[
(x-4)^2 + (y-4)^2 = 5,
\qquad
x^2 + y^2 = 16.
\]

Como as curvas são duas circunferências, existe interseção em dois pontos.  
Vou usar Newton para sistemas a partir de duas aproximações iniciais:
- \((3{,}8, 1{,}2)\);
- \((1{,}2, 3{,}8)\).

In [5]:
F = lambda v: np.array([
    (v[0] - 4)**2 + (v[1] - 4)**2 - 5,
    v[0]**2 + v[1]**2 - 16
], dtype=float)

J = lambda v: np.array([
    [2*(v[0] - 4), 2*(v[1] - 4)],
    [2*v[0],       2*v[1]]
], dtype=float)

for x0 in ([3.8, 1.2], [1.2, 3.8]):
    raiz, it, _ = newton_system(F, J, x0, tol=1e-6)
    print(f"x0 = {x0} -> raiz ≈ {raiz} | iterações = {it}")

x0 = [3.8, 1.2] -> raiz ≈ [3.5691709988 1.8058290012] | iterações = 4
x0 = [1.2, 3.8] -> raiz ≈ [1.8058290012 3.5691709988] | iterações = 4


## Questão 3

**Enunciado reescrito.** Encontrar o ângulo de disparo \(\alpha\) que produz tensão média de saída de 100 V:
\[
T_m = \frac{T_{max}}{\pi}(1 + \cos\alpha),
\qquad
T_{max} = 220\sqrt{2}.
\]

A equação de raiz fica:
\[
\frac{T_{max}}{\pi}(1 + \cos\alpha) - 100 = 0.
\]

Vou trabalhar com \(\alpha\) em **radianos** durante o cálculo e converter para graus no final.

In [6]:
Tmax = 220*np.sqrt(2)

f = lambda a: Tmax/np.pi * (1 + np.cos(a)) - 100
df = lambda a: -Tmax/np.pi * np.sin(a)

alpha_rad, it, _ = newton(f, df, x0=1.0, tol=1e-6)
alpha_deg = np.degrees(alpha_rad)

print(f"alpha ≈ {alpha_rad:.12f} rad")
print(f"alpha ≈ {alpha_deg:.12f} graus")
print(f"iterações = {it}")

alpha ≈ 1.561050050226 rad
alpha ≈ 89.441579486607 graus
iterações = 3


## Questão 4

**Enunciado reescrito.** Determinar a indutância \(L\) de um filtro LC passa-baixa com
\[
f_c = \frac{1}{2\pi\sqrt{LC}},
\]
sabendo que:
- \(f_c = 1000\) Hz;
- \(C = 1\,\mu F\).

A raiz será procurada em:
\[
g(L) = \frac{1}{2\pi\sqrt{LC}} - 1000.
\]

In [7]:
C = 1e-6
fc = 1000.0

g = lambda L: 1/(2*np.pi*np.sqrt(L*C)) - fc
dg = lambda L: -1/(4*np.pi*np.sqrt(C)*L**(3/2))

L_sol, it, _ = newton(g, dg, x0=0.01, tol=1e-6)

print(f"L ≈ {L_sol:.12f} H")
print(f"iterações = {it}")

# só para conferência
L_exata = 1/(((2*np.pi*fc)**2) * C)
print(f"valor de conferência = {L_exata:.12f} H")

L ≈ 0.025330295906 H
iterações = 5
valor de conferência = 0.025330295911 H


## Questão 5

**Enunciado reescrito.** No circuito RLC série com perdas, encontrar a frequência real de ressonância usando:
- Newton;
- secante;
- secante modificada.

A condição de ressonância é a anulação da parte imaginária da impedância:
\[
\omega L - \frac{1}{\omega C} = 0,
\qquad \omega = 2\pi f.
\]

Logo a função em termos de \(f\) é:
\[
h(f) = 2\pi f L - \frac{1}{2\pi f C}.
\]

In [8]:
R = 10.0
L = 1e-3
C = 1e-6

h = lambda f: 2*np.pi*f*L - 1/(2*np.pi*f*C)
dh = lambda f: 2*np.pi*L + 1/(2*np.pi*C*f**2)

f_newton, it_newton, _ = newton(h, dh, x0=5000, tol=1e-6)
f_sec, it_sec, _ = secant(h, 4000, 6000, tol=1e-6)
f_mod, it_mod, _ = modified_secant(h, 5000, delta=0.01, tol=1e-6)

f_ideal = 1/(2*np.pi*np.sqrt(L*C))

print(f"Newton             -> f ≈ {f_newton:.12f} Hz | iterações = {it_newton}")
print(f"Secante            -> f ≈ {f_sec:.12f} Hz | iterações = {it_sec}")
print(f"Secante modificada -> f ≈ {f_mod:.12f} Hz | iterações = {it_mod}")
print(f"Frequência ideal   -> f0 = {f_ideal:.12f} Hz")

print("\nComentário:")
print("- Neste modelo, a condição de ressonância cai na mesma equação ideal da parte imaginária nula.")

Newton             -> f ≈ 5032.921209281776 Hz | iterações = 2
Secante            -> f ≈ 5032.921288328312 Hz | iterações = 4
Secante modificada -> f ≈ 5032.921211835011 Hz | iterações = 3
Frequência ideal   -> f0 = 5032.921210448703 Hz

Comentário:
- Neste modelo, a condição de ressonância cai na mesma equação ideal da parte imaginária nula.


## Questão 6

**Enunciado reescrito.** Determinar a frequência \(F\) em dois métodos diferentes para o circuito RLC com:
- \(R = 140\ \Omega\),
- \(L = 260\,mH\),
- \(C = 25\,\mu F\),
- \(v_m = 24\ V\),
- \(i_m = 0{,}15\ A\),
- chute inicial \(x_0 = 30\).

A equação de raiz fica:
\[
\frac{v_m}{\sqrt{R^2 + \left(2\pi F L - \frac{1}{2\pi F C}\right)^2}} - i_m = 0.
\]

Como o enunciado só fornece um chute inicial, vou usar:
- Newton;
- secante modificada.

In [9]:
R = 140.0
L = 260e-3
C = 25e-6
vm = 24.0
im = 0.15

f = lambda F: vm/np.sqrt(R**2 + (2*np.pi*F*L - 1/(2*np.pi*F*C))**2) - im
df = lambda F: -vm*(2*(2*np.pi*F*L - 1/(2*np.pi*F*C))*(2*np.pi*L + 1/(2*np.pi*C*F**2))) / (2*(R**2 + (2*np.pi*F*L - 1/(2*np.pi*F*C))**2)**(3/2))

F_newton, it_newton, _ = newton(f, df, x0=30, tol=1e-6)
F_mod, it_mod, _ = modified_secant(f, 30, delta=0.01, tol=1e-6)

print(f"Newton             -> F ≈ {F_newton:.12f} Hz | iterações = {it_newton}")
print(f"Secante modificada -> F ≈ {F_mod:.12f} Hz | iterações = {it_mod}")

print("\nObservação:")
print("- Com x0 = 30 Hz, os dois métodos convergem para a raiz de baixa frequência.")

Newton             -> F ≈ 43.067957905835 Hz | iterações = 3
Secante modificada -> F ≈ 43.068109413986 Hz | iterações = 4

Observação:
- Com x0 = 30 Hz, os dois métodos convergem para a raiz de baixa frequência.


## Questão 7

**Enunciado reescrito.** Determinar \(V_{mp}\) na equação da célula solar:
\[
e^{\left(\frac{qV_{mp}}{k_B T}\right)}
\left(1 + \frac{qV_{mp}}{k_B T}\right)
=
e^{\left(\frac{qV_{oc}}{k_B T}\right)}.
\]

Vou usar Newton-Raphson.

In [11]:
q = 1.6022e-19
kB = 1.3806e-23
Voc = 0.5
T = 297.0

f = lambda V: np.exp(q*V/(kB*T))*(1 + q*V/(kB*T)) - np.exp(q*Voc/(kB*T))
df = lambda V: np.exp(q*V/(kB*T))*(q/(kB*T))*(2 + q*V/(kB*T))

Vmp, it, _ = newton(f, df, x0=0.4, tol=1e-6)

print(f"Vmp ≈ {Vmp:.12f} V")
print(f"iterações = {it}")

Vmp ≈ 0.426508971804 V
iterações = 6


## Questão 8

**Enunciado reescrito.** Resolver o sistema não linear do painel usando **as funções de iteração fornecidas** e os valores iniciais:
- \(T_h = T_c = 298\ K\),
- \(J_c = 3000\ W/m^2\),
- \(J_h = 5000\ W/m^2\).

Vou organizar o vetor iterado na ordem:
\[
[T_c, T_h, J_c, J_h].
\]

In [12]:
def G(v):
    Tc, Th, Jc, Jh = v
    Tc_new = ((Jc - 17.41*Tc + 5188.18)/(5.67e-8))**0.25
    Th_new = ((2250 + Jh - 1.865*Th)/(5.67e-8))**0.25
    Jc_new = 2352.71 + 0.71*Jh - 7.46*Tc
    Jh_new = 11093 + 0.71*Jc - 7.46*Th
    return np.array([Tc_new, Th_new, Jc_new, Jh_new], dtype=float)

sol, it, _ = fixed_point_system(G, [298, 298, 3000, 5000], tol=1e-6, maxit=10000)
Tc, Th, Jc, Jh = sol

print("Solução aproximada:")
print(f"Tc ≈ {Tc:.12f} K")
print(f"Th ≈ {Th:.12f} K")
print(f"Jc ≈ {Jc:.12f} W/m²")
print(f"Jh ≈ {Jh:.12f} W/m²")
print(f"iterações = {it}")

res = np.array([
    5.67e-8*Tc**4 + 17.41*Tc - Jc - 5188.18,
    Jc - 0.71*Jh + 7.46*Tc - 2352.71,
    5.67e-8*Th**4 + 1.865*Th - Jh - 2250,
    Jh - 0.71*Jc + 7.46*Th - 11093
], dtype=float)

print("\nResíduo do sistema original:")
print(res)

Solução aproximada:
Tc ≈ 481.027254636360 K
Th ≈ 671.123977944654 K
Jc ≈ 6222.225082071500 W/m²
Jh ≈ 10504.194933200746 W/m²
iterações = 83

Resíduo do sistema original:
[-0.0000022978 -0.0000009138  0.0000003455  0.0000003971]
